# 6. Воспроизводимый отчёт

**Цели:** собрать результаты в один отчёт с источником данных, конфигурацией модели, численными артефактами и оговорками. Сетевые ответы не должны быть единственным хранилищем результата.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/fractal_extreme_series.csv").exists())
DATA = ROOT / "data/fractal_extreme_series.csv"
SEED = 42
rng = np.random.default_rng(SEED)
df = pd.read_csv(DATA)
x = df["value"].to_numpy()
assert np.isfinite(x).all() and x.ndim == 1
# Graph-EVT-agent is the analysis library; no model code is copied into this repository.
from graph_evt_agent import ResearchAgent, ResearchTask, LLMConfig

llm = LLMConfig.from_env()  # provider/model/key are selected with environment variables
agent = ResearchAgent(llm=llm, seed=SEED)


In [ ]:
task=ResearchTask(
 question="Сформируй краткий учебный отчёт об экстремумах и границах выводов", data=x,
 methods=["pot","declustering","return_levels","sensitivity_analysis"],
 constraints={"seed":SEED,"include_provenance":True,"include_uncertainty":True,"language":"ru"},
)
report=agent.run(task)
out=ROOT/'results/06_report'; report.save(out)
report.summary


In [ ]:
# Эти поля должны присутствовать до публикации отчёта.
required={'data_hash','library_version','llm_model','parameters','tables','limitations'}
missing=required-set(report.manifest)
assert not missing, f'Неполный manifest: {sorted(missing)}'
report.manifest


Сравните повторный запуск с тем же seed и другой LLM-конфигурацией. Текст может меняться; входной hash, параметры и рассчитанные таблицы должны оставаться аудируемыми.

**Самопроверка:** какие части детерминированы? Почему фиксированный seed недостаточен для LLM? Что необходимо архивировать? Какие выводы нельзя переносить за пределы этого ряда?